# 02 · Feature engineering
Reuniwatt + computed features with the three lag families (around the target, across runs, across horizons), plus labels.

Settings come from `conf/parameters.yml`; the widgets below override the `run` section. Outputs are listed in `conf/catalog.yml`. If a node fails, fix it and set `from_node` to resume.

In [ ]:
%pip install -q pyyaml lightgbm
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("output_schema", "ewec_dev_powerops.scratch_yash", "Output schema")
dbutils.widgets.text("product", "dayahead_15min", "Product")
dbutils.widgets.text("start_date", "2024-01-23", "First origin date")
dbutils.widgets.text("end_date", "", "Last origin date (blank = yesterday)")
dbutils.widgets.text("from_node", "", "Resume from node (blank = start)")

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(REPO, "src"))
for m in [m for m in list(sys.modules) if m.startswith("uc2_solar")]:
    del sys.modules[m]          # reload code after a git pull
from uc2_solar.runner import run
from uc2_solar.registry import register_pipelines

OVERRIDES = {k: dbutils.widgets.get(k) for k in ["output_schema", "product", "start_date", "end_date"]}
FROM_NODE = dbutils.widgets.get("from_node").strip() or None
print("Repo:", REPO)

## Pipeline
The nodes that will run, with their inputs and outputs:

In [ ]:
print(register_pipelines()["feature_engineering"].describe())

## Run

In [ ]:
timings = run("feature_engineering", spark, REPO, OVERRIDES, base_dir=os.getcwd(), from_node=FROM_NODE)
timings

✅ **Feature table size and coverage**: rows ≈ days × 2 origins × 96 steps × 2 sites; forecast coverage near 100%.

In [ ]:
S = dbutils.widgets.get("output_schema"); P = dbutils.widgets.get("product")
from pyspark.sql import functions as F
ft = spark.table(f"{S}.uc2_feat_{P}")
print(len(ft.columns), "columns")
display(ft.groupBy("site").agg(F.count("*").alias("rows"), F.countDistinct("origin").alias("origins"), F.round(100 * F.avg(F.col("f_power_mw").isNotNull().cast("int")), 1).alias("pct_forecast"), F.round(100 * F.avg(F.col("label_ok").cast("int")), 1).alias("pct_label_ok")))

✅ **Null % per feature** (top 40): `wa_*` and across-runs columns may be higher; most should be low.

In [ ]:
nulls = ft.select(*[F.round(100 * F.avg(F.col(c).isNull().cast("int")), 1).alias(c) for c in ft.columns])
row = nulls.first().asDict()
import pandas as pd
display(pd.Series(row, name="null_pct").sort_values(ascending=False).head(40).reset_index())